# EEG controls, phase 1, on KG-GT v2

**Where we are.** In `main_v2.ipynb` E (the EEG-only output) reaches r = 0.645, but F = K: the EEG adds almost nothing to the
kinematics. The first three controls (`eeg_controls.ipynb`) showed that E mostly knows *when* a lift happens, that its
electrode map is partly sensorimotor and partly eye and neck, and that the EEG helps a little around reaching when the
envelope is forecast. The long-term goal is an E that gives each lift the right **level** (heavy higher than light).
Phase 1 checks the current preprocessing before anything is changed.

| section | question | trains a network? |
|---|---|---|
| **W. Weight check** | Does E already give different levels for light, medium and heavy lifts, and is it only too low or flat? | no (about 1 min per participant) |
| **R. Ridge per phase and timing baseline** | Does the EEG add anything beyond knowing the times of past task events, and does a ridge per task phase help? | no (about 3 min per participant and horizon) |
| **C. Electrode subsets and eye control** | How much of E and of the EEG gain survives with only central electrodes, only edge electrodes, or the eyes regressed out? | yes (E and fusion only, K loaded and frozen; about 10-15 min per run, 108 runs) |

Every section has a RUN cell (computes and saves tables) and a PLOT cell (reads the saved tables, so it can be re-run alone).
Tables go to `eeg_controls/results/phase1/`, figures to `eeg_controls/results/phase1/figures/`, the condition runs to
`results/v2_controls/` and `outputs/checkpoints_v2_controls/`. Read only: the cache, the raw files, `outputs/checkpoints_v2/`,
`outputs/checkpoints_v2_horizon/`, `results/v2_horizon/`. The code is in `eeg_controls/` (see `eeg_controls/README.md`); this
notebook is generated by `tools/build_eeg_controls_phase1_notebook.py`. The design is in `eeg_controls/DESIGN_phase1.md`.

In [ ]:
# P1-SETUP
import time
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from eeg_controls import weight_check as WC, phase_ridge as PR, run_conditions as RC, plots_phase1 as PP
from eeg_controls.common import parse_subjects
from v2.interp import plots as P

PARTICIPANTS = parse_subjects("all")          # e.g. "1-4" or [1, 2] for a quick look
OUT = Path("eeg_controls/results/phase1")
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| participants:", PARTICIPANTS)


def show(fig, name):
    """Save the figure to eeg_controls/results/phase1/figures/<name>.png and show it."""
    with P.style():
        fig.savefig(FIG / f"{name}.png", dpi=150)
    display(fig)
    plt.close(fig)

## W. Weight check: does E give heavy lifts a higher level than light ones?

For every test lift with a known weight (CurW 1, 2, 4 = light, medium, heavy), the **level** is the mean envelope of the
target and of each output (K, E, F) per muscle in two windows: *hold* (lift-off + 0.5 s to replace - 0.2 s, the main window)
and *load* (first touch to lift-off). The checkpoints are the non-causal v2 models in `outputs/checkpoints_v2`.
**E_cal** is E multiplied by one gain per muscle, fitted on the validation lifts: it shows how much of the level error a
single rescaling fixes, and whether a weight difference shows once the size is right.

**What the numbers mean.** `rho_weight` is the Spearman correlation of the lift level with the weight (the target's is the
reference). `ratio_heavy_light` is the mean level of heavy over light lifts. `slope` is the slope of output level on target
level over lifts (1 = right size, below 1 = shrunk toward the mean). `size_ratio` is mean output over mean target.

**How to read the figures.**

- **W1** (mean level per weight, per muscle): the target rises from light to heavy. A *good for EEG* E rises too, even if it is
  too low; E_cal then sits on the target. If E is *flat* across weights while E_cal is on the target only on average, the EEG
  carries timing but no force information.
- **W2** (lift by lift, E against target level, colours = weight): good is a cloud along the identity line, with dark (heavy)
  points above light ones. Only timing looks like a horizontal band: all lifts get the same level whatever the target.
- **W3** (a stretch of one test series with lifts of different weights): look at whether E's bursts differ in height between
  light and heavy lifts, and whether E_cal follows the target's height.
- **W4** (per participant): rho and slope for E and K. Good is rho of E clearly above 0 in most participants and a slope
  above 0; K is the reference for what the kinematics give.

The printout after the figures gives E's `rho_weight` and heavy/light ratio against the target's, and the group tests over
participants (Wilcoxon).

In [ ]:
# P1-W-RUN   (no training: about 1 min per participant)
W_CKPT = Path("outputs/checkpoints_v2")
(OUT / "weight").mkdir(parents=True, exist_ok=True)
lv, st, example = [], [], None
for p in PARTICIPANTS:
    t0 = time.time()
    res = WC.run_participant(p, W_CKPT, keep_example=(p == PARTICIPANTS[0]))
    lv.append(res[0]); st.append(res[1])
    if p == PARTICIPANTS[0]:
        example = res[2]
    print(f"P{p} done ({time.time() - t0:.0f} s)")
levels, stats = pd.concat(lv, ignore_index=True), pd.concat(st, ignore_index=True)
group = WC.group_tests(stats)
levels.to_csv(OUT / "weight" / "levels.csv", index=False)
stats.to_csv(OUT / "weight" / "stats.csv", index=False)
group.to_csv(OUT / "weight" / "group.csv", index=False)
print()
print(group.round(4).to_string(index=False))

In [ ]:
# P1-W-PLOT
levels = pd.read_csv(OUT / "weight" / "levels.csv")
stats = pd.read_csv(OUT / "weight" / "stats.csv")
group = pd.read_csv(OUT / "weight" / "group.csv")
show(PP.weight_levels(stats), "W1_level_per_weight")
show(PP.weight_scatter(levels), "W2_level_scatter")
if "example" in globals() and example is not None:           # needs the RUN cell of this session
    p0 = int(levels.participant.iloc[0])
    gain = PP.gain_from_levels(levels[levels.participant == p0])
    show(PP.weight_example(example, names=example["names"], gain=gain), "W3_example_series")
else:
    print("W3 needs the RUN cell of this session (the example series is not saved); skipped.")
show(PP.weight_participants(stats), "W4_per_participant")

hold = stats[stats.window == "hold"]
per_p = hold.groupby(["output", "participant"])[["rho_weight", "ratio_heavy_light", "slope", "size_ratio"]].mean()
mean = per_p.groupby("output").mean().reindex(["target", "K", "E", "E_cal", "F"]).dropna(how="all")
print("hold window, mean over muscles then over participants:")
print(mean.round(3).to_string())
print()
print("E against the target: rho_weight", f"{mean.loc['E', 'rho_weight']:.3f} (target {mean.loc['target', 'rho_weight']:.3f}),",
      "heavy/light", f"{mean.loc['E', 'ratio_heavy_light']:.2f} (target {mean.loc['target', 'ratio_heavy_light']:.2f})")
print()
print("group tests over participants (rho_E_gt_0: E's rho > 0; ratio_E_lt_target: target ratio minus E ratio > 0):")
print(group.round(4).to_string(index=False))
print("W1/W2: levels per weight and lift by lift. W3: a real stretch. W4: each participant's rho and slope.")

## R. Ridge per phase and the causal timing baseline

The residual of K (what the kinematics miss) is predicted by a ridge regression on the validation series and scored on the
test series, at token rate, from the causal horizon checkpoints (`outputs/checkpoints_v2_horizon`, h = 0, 200, 500 ms).
Three feature sets, all causal (nothing depends on a later token):

- **eeg**: the EEG tokens of the trained E stream.
- **events** (the timing baseline): past-event bumps. For each of the 8 task events that has already happened at or before the
  token, Gaussian bumps at lags 0 to +3 s after it. It knows everything a timing model could know at that moment.
- **both**: the two together.

Two ways to fit: **pooled** (one ridge over all task phases) and **per_phase** (one ridge per phase: pre_move, reach, load,
hold, release, other, fitted on the validation tokens of that phase only). Scores per phase: `skill` = 1 - MSE(K + prediction)
/ MSE(K) (above 0 = the prediction removes error), r with the residual, and a time-shift null. The key number is
`gain_both_minus_events` = skill(both) - skill(events): what the EEG adds **beyond timing**, with a Wilcoxon test over
participants.

**How to read the figures.**

- **R1** (heat maps phase by horizon, one panel per feature set and fit): look at the `events` panels first. Where they are
  already warm, timing explains the gain. A *good for EEG* result is `both` clearly warmer than `events` in pre_move and
  reach, increasing with the horizon. *Only timing* looks like `eeg`, `events` and `both` all alike.
- **R2** (skill against horizon for pre_move and reach, lines for eeg, events, both): the gap between `both` and `events` is
  the EEG gain beyond timing. Lines on top of each other mean the EEG adds nothing the event times do not.
- **R3** (per-participant gain beyond timing, per phase): good is most dots above 0 in pre_move and reach. A cloud around 0 or
  one participant carrying the mean means the result rests on few participants.

The per_phase fit may help or hurt: with few tokens per phase it overfits, so compare it with pooled in the same panel.

In [ ]:
# P1-R-RUN   (no training: about 3 min per participant and horizon)
R_CKPT = Path("outputs/checkpoints_v2_horizon")
R_HORIZONS = [0, 200, 500]
R_NULL = 99                                  # time-shift null draws
(OUT / "phase_ridge" / "runs").mkdir(parents=True, exist_ok=True)
tables = []
for h in R_HORIZONS:
    for p in PARTICIPANTS:
        f = OUT / "phase_ridge" / "runs" / f"h{h}ms_P{p}.csv"       # finished runs are reused
        if f.exists():
            df = pd.read_csv(f)
        else:
            t0 = time.time()
            df = PR.run_participant(p, h, R_CKPT, n_null=R_NULL)
            df.to_csv(f, index=False)
            print(f"P{p} h={h} ms done ({time.time() - t0:.0f} s)")
        tables.append(df)
phase_ridge = pd.concat(tables, ignore_index=True)
phase_ridge.to_csv(OUT / "phase_ridge" / "phase_ridge.csv", index=False)
summary = PR.summarise(phase_ridge)
summary.to_csv(OUT / "phase_ridge" / "summary.csv", index=False)
k = summary[summary.segment.isin(["pre_move", "reach", "all"]) & (summary.features == "both")]
print(k[["horizon_ms", "segment", "fit", "n", "gain_both_minus_events", "p_both_gt_events"]].round(4).to_string(index=False))

In [ ]:
# P1-R-PLOT
summary = pd.read_csv(OUT / "phase_ridge" / "summary.csv")
phase_ridge = pd.read_csv(OUT / "phase_ridge" / "phase_ridge.csv")
show(PP.ridge_heatmaps(summary), "R1_skill_heatmaps")
show(PP.ridge_lines(summary), "R2_skill_vs_horizon")
show(PP.ridge_gain(phase_ridge, "per_phase"), "R3_gain_beyond_timing_per_phase")

key = summary[(summary.features == "both") & summary.segment.isin(["pre_move", "reach", "all"])]
print("EEG gain beyond timing = skill(both) - skill(events), mean over participants, Wilcoxon p (one-sided):")
for fit in ("pooled", "per_phase"):
    t = key[key.fit == fit].pivot_table(index="horizon_ms", columns="segment",
                                        values=["gain_both_minus_events", "p_both_gt_events"])
    print(f"\nfit: {fit}")
    print(t.round(4).to_string())
sk = summary[summary.segment.isin(["pre_move", "reach"])].pivot_table(index=["horizon_ms", "segment"],
                                                                      columns=["features", "fit"], values="skill_mean")
print("\nmean skill per phase, feature set and fit (per_phase against pooled):")
print(sk.round(4).to_string())
print("R1: skill per phase and horizon. R2: the gap between both and events is the gain beyond timing. R3: per participant.")

## C. Electrode subsets and eye control: what survives in E and in the EEG gain?

Three conditions, each at h = 0, 200, 500 ms and for every participant, compared with the existing all-electrode horizon runs
(`all`, from `results/v2_horizon`):

- **central** (7 electrodes: C3, Cz, C4, CP1, CP2, CP5, CP6): the sensorimotor strip.
- **edge** (15 electrodes: Fp1, Fp2, F7, F8, T7, T8, TP9, TP10, P7, P8, PO9, PO10, O1, Oz, O2): the rim, where eyes, neck and
  muscles show up. It has more channels than central, so a gain of edge over central is not only about size.
- **eye_clean**: Fp1, Fp2, F7, F8 are used as eye references. Every other channel is regressed on them (per series, least
  squares) and the references are dropped, leaving 28 channels. The weights are a fixed spatial filter, so no future value
  enters a token.

**K is loaded from the matching horizon checkpoint and frozen from the first epoch; only E and the fusion train.** So every
difference to `all` comes from the EEG input alone. Train this cell yourself: it needs the horizon checkpoints in
`outputs/checkpoints_v2_horizon` and takes about 10-15 min per run (3 conditions x 3 horizons x 12 participants = 108 runs, about
18-27 h). Finished runs are skipped, an interrupted run resumes. Start with `C_PARTICIPANTS = [1, 2]`.

**How to read the figures.** Measures: `E_r` (accuracy of E), `F_minus_K_vaf` (what fusion adds to K), `res_r_tokens` (correlation
of the prediction with the residual of K), `pre_move_*` and `reach_*` r and skill.

- **C1** (E r and F - K per condition and horizon): if `central` keeps E r close to `all` while `edge` falls, E lives in
  sensorimotor channels. If `edge` keeps most of it, E uses eyes, neck or timing artifacts. A fall of `eye_clean` shows how
  much of E was eye movement.
- **C2** (per-phase residual r per condition): the EEG gain should survive in `central` and `eye_clean` at pre_move and reach and
  grow with the horizon. If it only survives in `edge`, or disappears with `eye_clean`, treat it as an artifact.
- **C3** (per participant, condition against `all`): dots on the diagonal mean the condition changes nothing; below it, the
  condition removes information. A few outliers carrying the mean mean it is not a group result.

The printout lists condition minus `all` per measure with the two-sided paired Wilcoxon p over participants.

In [ ]:
# P1-C-RUN   (TRAINS: about 10-15 min per run, 108 runs for 12 participants; needs outputs/checkpoints_v2_horizon)
C_PARTICIPANTS = PARTICIPANTS            # e.g. [1, 2] first
CONDITIONS = ["central", "edge", "eye_clean"]
HORIZONS_MS = [0, 200, 500]
args = SimpleNamespace(results="results/v2_controls", checkpoints="outputs/checkpoints_v2_controls",
                       kin_checkpoints="outputs/checkpoints_v2_horizon", baseline="results/v2_horizon",
                       smoke=False, amp=False, overwrite=False)
for h in HORIZONS_MS:
    for name in CONDITIONS:
        for p in C_PARTICIPANTS:
            RC.run_one(p, name, h, args, device)
summary = RC.summarise(Path(args.results), Path(args.baseline))

In [ ]:
# P1-C-PLOT
C_RESULTS = Path("results/v2_controls")
C_BASELINE = Path("results/v2_horizon")
summary = RC.summarise(C_RESULTS, C_BASELINE)
if summary.empty:
    print("no finished runs yet: run the P1-C-RUN cell first")
else:
    show(PP.conditions_overview(summary), "C1_E_and_fusion_per_condition")
    show(PP.conditions_phases(summary), "C2_per_phase_per_condition")
    show(PP.conditions_participants(summary), "C3_per_participant_vs_all")
    vs = pd.read_csv(C_RESULTS / "vs_all.csv")
    key = vs[vs.metric.isin(["E_r", "F_minus_K_vaf", "res_r_tokens", "pre_move_skill"])]
    print("\ncondition minus all, mean over participants, and two-sided Wilcoxon p:")
    print(key.pivot_table(index=["condition", "horizon_ms"], columns="metric", values=["mean_diff", "p"], sort=False)
          .round(4).to_string())
    print("\nC1: E r and F - K per condition. C2: per-phase residual r. C3: each participant, condition against all.")